### Data Ingestion

In [5]:
from langchain_core.documents import Document

In [9]:
doc = Document(
  page_content="This is a test document.", 
  metadata={
    "source": "test", 
    "author": "John Doe",
    }
  )
doc

Document(metadata={'source': 'test', 'author': 'John Doe'}, page_content='This is a test document.')

In [10]:
## create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [11]:
sample_texts = {
"../data/text_files/python_intro.txt": "Python is a high-level, interpreted programming language known for its readability and versatility.",
"../data/text_files/machine_learning.txt": "Machine learning is a field of artificial intelligence that uses algorithms to learn patterns from data."
}

for file_name, content in sample_texts.items():
  with open(file_name, "w", encoding="utf-8") as f:
    f.write(content)

In [15]:
## TextLoader
# from langchain_core.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt")
data = loader.load()
print(data)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its readability and versatility.')]


In [16]:
## Directory Loader

from langchain_community.document_loaders import DirectoryLoader

directory_loader = DirectoryLoader(
"../data/text_files",
glob="**/*.txt", #A pattern used to match specific file types (e.g., "**/*.txt" for all text files).
loader_cls=TextLoader, 
loader_kwargs={"encoding": "utf-8"},
show_progress=False # Set to False in the video to avoid needing extra dependencies like Tqdm
)

documents=directory_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine learning is a field of artificial intelligence that uses algorithms to learn patterns from data.'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming language known for its readability and versatility.')]

In [19]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

directory_loader = DirectoryLoader(
"../data/pdf",
glob="**/*.pdf", #A pattern used to match specific file types (e.g., "**/*.pdf" for all pdf files).
loader_cls=PyMuPDFLoader, 
show_progress=False # Set to False in the video to avoid needing extra dependencies like Tqdm
)

pdf_documents=directory_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2026-10-19T00:07:48+05:00', 'source': '..\\data\\pdf\\hackathonguide.pdf', 'file_path': '..\\data\\pdf\\hackathonguide.pdf', 'total_pages': 8, 'format': 'PDF 1.7', 'title': 'Microsoft Word - HackInsight_Participant_Guide_7pages', 'author': 'fizzah', 'subject': '', 'keywords': '', 'moddate': '2026-10-19T00:07:48+05:00', 'trapped': '', 'modDate': "D:20261019000748+05'00'", 'creationDate': "D:20261019000748+05'00'", 'page': 0}, page_content='How to use this PDF \nBlue underlined text is a clickable link, both inside the guide and to websites. Click any entry in the Contents, or \'Go to Contents\' at the top of a page, to jump. \nOpen the bookmarks panel of your PDF viewer to move between chapters. Tick the boxes and type in the blue fill-in fields. Use Adobe Acrobat, Chrome or \nEdge so that your answers are kept when you save the file. \nEach chapter also appears in the bookmarks panel. Type your a

In [20]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### Embeddings and VectorStoreDB

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Tesla Laptops\Downloads\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:

class EmbeddingManager:
  def __init__(self, model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    self.model = None
    self._load_model()

  def _load_model(self):
    try:
      print(f"Loading embedding model : {self.model_name}...")
      self.model = SentenceTransformer(self.model_name)
      print(f"{self.model_name} model loaded successfully.")
      print(f"Dimension: {self.model.get_embedding_dimension()}")
    except Exception as e:
      print(f"Error loading model {self.model_name}: {e}")
      raise

  def generate_embeddings(self, text: List[str]) -> np.ndarray:
      if not self.model:
          raise ValueError("Model is not loaded. Please check the model initialization.")
      print(f"Generating Embeddings for {len(text)} texts...")
      embeddings =  self.model.encode(text, show_progress_bar=True)
      print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
      return embeddings

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model : all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1829.89it/s]


all-MiniLM-L6-v2 model loaded successfully.
Dimension: 384


### VectorStore

In [25]:
class VectorStore:
  def __init__(self, collection_name="pdf_documents", persistent_directory="../data/vector_store"):
    self.collection_name = collection_name
    self.persistent_directory = persistent_directory
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
    try:
      print(f"Initializing vector store at '{self.persistent_directory}'...")
      # Initialize chromadb client and collection
      os.makedirs(self.persistent_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persistent_directory)
      # Get or create the collection
      self.collection = self.client.get_or_create_collection(name=self.collection_name)
      print(f"Collection '{self.collection_name}' initialized. Existing document count: {self.collection.count()}")
    except Exception as e:
        print(f"Error initializing vector store: {e}")
        raise

  def add_documents(self, documents, embeddings):
      # Add documents and the embeddings to the vector store
      if len(documents) != len(embeddings):
          raise ValueError("Number of documents and embeddings must match.")
      print(f"Adding {len(documents)} documents to the collection '{self.collection_name}'...")

      # Prepare data for chromaDB
      ids = []
      metadatas = []
      document_texts = []
      embedding_list = []

      for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
          doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
          # Generate a unique ID for each document using UUID and the index to ensure uniqueness
          ids.append(doc_id)

          # prepare metadata
          metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
          metadata['doc_index'] = i  # Add the index to the metadata for reference
          metadata['content_length'] = len(doc.page_content)  # Add content length to metadata

          metadatas.append(metadata)

          # prepare document content
          document_texts.append(doc.page_content)
          # Embedding should be converted to a list for storage in ChromaDB
          embedding_list.append(embedding.tolist())

      # Add to collection
      try:
        self.collection.add(
            ids=ids,
            embeddings=embedding_list,
            metadatas=metadatas,
            documents=document_texts
        )
        print(f"Added {len(documents)} documents to the collection.")
        print(f"Total documents in collection '{self.collection_name}': {self.collection.count()}")
      except Exception as e:
        print(f"Error adding documents to the collection: {e}")
        raise

vector_store = VectorStore()
vector_store

Initializing vector store at '../data/vector_store'...
Collection 'pdf_documents' initialized. Existing document count: 0
